# 401 — Step Result Contract Check

Verifies that `OrchestratorResult.step_results` now expose the full UI contract needed by Phase 4.2:

| Field | Type | Description |
|---|---|---|
| `step_id` | str | Stable plan step identifier |
| `agent` | str | Agent that executed the step |
| `task` | str | Task name dispatched |
| `status` | str | `success` / `failed` / `skipped` |
| `tools_executed` | list[str] | MCP tools called by the agent |
| `summary` | str | Short human-readable output |
| `findings` | dict | Structured task output |
| `error` | str \| None | Error detail when status=failed |
| `skipped` | bool | Backward-compat flag |
| `skip_reason` | str \| None | Why the step was skipped |

Three flows are tested:
1. **Investigate** — normal entity investigation
2. **Trace replay** — retrieve a prior trace by entity name
3. **Unresolved entity** — a name that does not exist in the graph (all steps skipped)

In [1]:
import sys
sys.path.insert(0, "..")

import json
import pprint
from dotenv import load_dotenv
load_dotenv("../.env")

True

## Wire components

In [2]:
from src.config import get_neo4j_settings, get_anthropic_settings
from src.clients.anthropic_client import AnthropicClient
from src.clients.mcp_tool_client import MCPToolClient
from src.storage.neo4j_repository import Neo4jRepository
from src.storage.trace_repository import TraceRepository
from src.tracing.trace_service import TraceService
from src.agents.graph_agent import GraphAgent
from src.agents.risk_agent import RiskAgent
from src.agents.trace_agent import TraceAgent
from src.orchestration.planner import InvestigationPlanner
from src.orchestration.orchestrator import Orchestrator

neo4j_settings = get_neo4j_settings()
anthropic_settings = get_anthropic_settings()

ai_client   = AnthropicClient(anthropic_settings)
mcp_client  = MCPToolClient()
repo        = Neo4jRepository(**vars(neo4j_settings))
trace_repo  = TraceRepository(repo)
trace_svc   = TraceService(trace_repo)

graph_agent = GraphAgent(mcp_client, trace_svc, ai_client)
risk_agent  = RiskAgent(mcp_client, trace_svc, ai_client)
trace_agent = TraceAgent(mcp_client, trace_svc, ai_client)

planner       = InvestigationPlanner(ai_client)
orchestrator  = Orchestrator(
    planner, mcp_client,
    graph_agent, risk_agent, trace_agent,
    trace_svc, trace_repo, ai_client,
)
print("Components wired.")

Components wired.


## Helper — pretty-print step results

In [3]:
REQUIRED_FIELDS = {"step_id", "agent", "task", "status", "tools_executed", "summary"}

def print_step_results(result, label=""):
    """Print each step cleanly and verify required fields are present."""
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"  mode={result.mode}  success={result.success}  trace_id={result.trace_id}")
    print(f"  steps={len(result.step_results)}")
    print(f"{'='*60}")

    all_ok = True
    for i, sr in enumerate(result.step_results, start=1):
        d = sr.to_dict()
        missing = REQUIRED_FIELDS - d.keys()
        ok = not missing
        all_ok = all_ok and ok

        badge = "✓" if ok else "✗ MISSING: " + str(missing)
        print(f"\nStep {i}: {d['step_id']}  [{badge}]")
        print(f"  agent          : {d['agent']}")
        print(f"  task           : {d['task']}")
        print(f"  status         : {d['status']}")
        print(f"  tools_executed : {d['tools_executed']}")
        print(f"  summary        : {(d['summary'] or '')[:120]}")
        if d.get('error'):
            print(f"  error          : {d['error']}")
        if d.get('skip_reason'):
            print(f"  skip_reason    : {d['skip_reason']}")

    print(f"\n{'─'*60}")
    print(f"  final_answer: {result.final_answer[:200]}")
    if result.warnings:
        for w in result.warnings:
            print(f"  warning: {w}")
    if result.errors:
        for e in result.errors:
            print(f"  error: {e}")
    status = "ALL FIELDS PRESENT" if all_ok else "SOME FIELDS MISSING — see above"
    print(f"\n  Contract check: {status}")
    return all_ok

## Flow 1 — Investigate (normal entity)

In [4]:
# Replace with a company name that exists in your Neo4j database.
INVESTIGATE_QUERY = "Investigate GLOBAL METALS UK LTD for ownership risk"

result_investigate = orchestrator.run(INVESTIGATE_QUERY)
ok1 = print_step_results(result_investigate, "Flow 1 — Investigate")

[03/23/26 19:10:01] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=628231;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=428201;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/23/26 19:10:05] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=996142;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=579180;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/23/26 19:10:09] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=534350;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=64057;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   


  Flow 1 — Investigate
  mode=investigate  success=True  trace_id=cb5b4cea-c7d9-4068-bf4d-6705a3c094f4
  steps=3

Step 1: step_1  [✓]
  agent          : graph-agent
  task           : entity_lookup
  status         : success
  tools_executed : ['entity_lookup']
  summary        : Found 10 company match(es) for 'SYNERGY GLOBAL METALS UK LTD'. Top match: 'SYNERGY GLOBAL METALS UK LTD' (number: 124712

Step 2: step_2  [✓]
  agent          : graph-agent
  task           : expand_ownership
  status         : success
  tools_executed : ['expand_ownership']
  summary        : Synergy Global Metals UK Ltd is ultimately owned by individual Ramnaresh Sonee with one direct ownership layer between h

Step 3: step_3  [✓]
  agent          : risk-agent
  task           : summarize_risk_for_company
  status         : success
  tools_executed : ['address_risk_check', 'control_signal_check', 'industry_context_check', 'ownership_complexity_check']
  summary        : SYNERGY GLOBAL METALS UK LTD presents

In [5]:
# Raw dict output for inspection
print(json.dumps(result_investigate.to_dict(), indent=2, default=str))

{
  "mode": "investigate",
  "query": "Investigate GLOBAL METALS UK LTD for ownership risk",
  "trace_id": "cb5b4cea-c7d9-4068-bf4d-6705a3c094f4",
  "success": true,
  "planner_output": {
    "mode": "investigate",
    "reason": "Ownership and risk query for a company. Resolving canonical name first, then walking the ownership chain and synthesising all four risk signals.",
    "entities": [
      {
        "name": "GLOBAL METALS UK LTD",
        "type": "Company"
      }
    ],
    "plan": [
      {
        "step_id": "step_1",
        "agent": "graph-agent",
        "task": "entity_lookup",
        "parameters": {
          "name": "GLOBAL METALS UK LTD"
        }
      },
      {
        "step_id": "step_2",
        "agent": "graph-agent",
        "task": "expand_ownership",
        "parameters": {
          "company_name": "GLOBAL METALS UK LTD",
          "max_depth": 5
        }
      },
      {
        "step_id": "step_3",
        "agent": "risk-agent",
        "task": "summariz

## Flow 2 — Trace replay

In [6]:
# Use the trace_id from Flow 1 to trigger a trace-mode replay.
trace_id_from_flow1 = result_investigate.trace_id

TRACE_QUERY = f"Retrieve and summarise the investigation trace for {result_investigate.query.split()[-1]}"
print(f"Trace query: {TRACE_QUERY!r}")
print(f"Expected trace_id to appear in findings: {trace_id_from_flow1}")

result_trace = orchestrator.run(TRACE_QUERY)
ok2 = print_step_results(result_trace, "Flow 2 — Trace replay")

Trace query: 'Retrieve and summarise the investigation trace for risk'
Expected trace_id to appear in findings: cb5b4cea-c7d9-4068-bf4d-6705a3c094f4


[03/23/26 19:11:34] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=169446;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=162170;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/23/26 19:11:36] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=489748;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=978789;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   


  Flow 2 — Trace replay
  mode=trace  success=False  trace_id=6355a523-81f2-41b9-a267-c9cd3137ea7b
  steps=0

────────────────────────────────────────────────────────────
  final_answer: The investigation trace could not be retrieved because no specific investigation reference or case identifier was provided in your query. Please resubmit your request with the relevant investigation c

  Contract check: ALL FIELDS PRESENT


## Flow 3 — Unresolved entity (all steps skipped)

In [7]:
UNRESOLVED_QUERY = "Investigate ZZZZZ_NONEXISTENT_COMPANY_XYZ for ownership risk"

result_unresolved = orchestrator.run(UNRESOLVED_QUERY)
ok3 = print_step_results(result_unresolved, "Flow 3 — Unresolved entity")

[03/23/26 19:11:39] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=158262;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=224839;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/23/26 19:11:41] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=637834;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=268946;file:///Users/emilpastor/opt/miniconda3/envs/ai-governance/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   


  Flow 3 — Unresolved entity
  mode=investigate  success=False  trace_id=48b1e085-c991-4d3e-a531-bd4247f21a3b
  steps=3

Step 1: step_1  [✓]
  agent          : graph-agent
  task           : entity_lookup
  status         : skipped
  tools_executed : []
  summary        : 
  skip_reason    : Entity not resolved — all steps skipped.

Step 2: step_2  [✓]
  agent          : graph-agent
  task           : expand_ownership
  status         : skipped
  tools_executed : []
  summary        : 
  skip_reason    : Entity not resolved — all steps skipped.

Step 3: step_3  [✓]
  agent          : risk-agent
  task           : summarize_risk_for_company
  status         : skipped
  tools_executed : []
  summary        : 
  skip_reason    : Entity not resolved — all steps skipped.

────────────────────────────────────────────────────────────
  final_answer: The entity 'ZZZZZ_NONEXISTENT_COMPANY_XYZ' does not exist in the available compliance database, preventing the ownership risk investigation from

## Contract summary

In [8]:
print("\nContract check results:")
print(f"  Flow 1 (investigate)    : {'PASS' if ok1 else 'FAIL'}")
print(f"  Flow 2 (trace replay)   : {'PASS' if ok2 else 'FAIL'}")
print(f"  Flow 3 (unresolved)     : {'PASS' if ok3 else 'FAIL'}")

# Per-field verification across all runs
print("\nField presence across all step_results:")
all_steps = (
    result_investigate.step_results
    + result_trace.step_results
    + result_unresolved.step_results
)
for field in sorted(REQUIRED_FIELDS):
    present = all(field in sr.to_dict() for sr in all_steps)
    print(f"  {field:20s} {'✓' if present else '✗'}")

# Verify status values are in the expected set
valid_statuses = {"pending", "running", "success", "failed", "skipped"}
bad = [sr for sr in all_steps if sr.status not in valid_statuses]
print(f"\nStatus values valid   : {'✓' if not bad else f'✗ unexpected: {[sr.status for sr in bad]}'}")

# Verify tools_executed is always a list
bad_tools = [sr for sr in all_steps if not isinstance(sr.tools_executed, list)]
print(f"tools_executed is list: {'✓' if not bad_tools else f'✗ bad: {bad_tools}'}")

# Verify skipped steps have empty tools_executed
skipped = [sr for sr in all_steps if sr.skipped]
bad_skipped = [sr for sr in skipped if sr.tools_executed]
print(f"Skipped → tools=[]    : {'✓' if not bad_skipped else f'✗ non-empty: {bad_skipped}'}")

assert not bad, "Unexpected status values"
assert not bad_tools, "tools_executed must always be a list"
print("\nAll assertions passed.")


Contract check results:
  Flow 1 (investigate)    : PASS
  Flow 2 (trace replay)   : PASS
  Flow 3 (unresolved)     : PASS

Field presence across all step_results:
  agent                ✓
  status               ✓
  step_id              ✓
  summary              ✓
  task                 ✓
  tools_executed       ✓

Status values valid   : ✓
tools_executed is list: ✓
Skipped → tools=[]    : ✓

All assertions passed.


## Sample: expected step_results shape after refactor

```json
[
  {
    "step_id": "step_1",
    "agent": "graph-agent",
    "task": "company_profile",
    "status": "success",
    "success": true,
    "summary": "Global Metals UK Ltd is registered at 1 Example St, London. SIC: 24100.",
    "findings": { "company_profile": { ... } },
    "tools_executed": ["company_profile"],
    "error": null,
    "skipped": false,
    "skip_reason": null
  },
  {
    "step_id": "step_2",
    "agent": "risk-agent",
    "task": "summarize_risk_for_company",
    "status": "success",
    "success": true,
    "summary": "Ownership chain of 3 hops with corporate-only structure. Address risk MEDIUM...",
    "findings": {
      "ownership_complexity_check": { "risk_level": "MEDIUM", ... },
      "control_signal_check":       { "risk_level": "LOW", ... },
      "address_risk_check":          { "risk_level": "HIGH", ... },
      "industry_context_check":      { "risk_level": "MEDIUM", ... }
    },
    "tools_executed": [
      "address_risk_check",
      "control_signal_check",
      "industry_context_check",
      "ownership_complexity_check"
    ],
    "error": null,
    "skipped": false,
    "skip_reason": null
  },
  {
    "step_id": "step_3",
    "agent": "graph-agent",
    "task": "entity_lookup",
    "status": "skipped",
    "success": false,
    "summary": "",
    "findings": {},
    "tools_executed": [],
    "error": null,
    "skipped": true,
    "skip_reason": "Halted — step 'step_2' (summarize_risk_for_company) failed."
  }
]
```